# Task 4: General Health Query Chatbot

So the idea here is pretty simple on paper — build a chatbot that can answer everyday health questions ("what causes a sore throat", "is paracetamol okay for kids") without it accidentally turning into something that gives risky medical advice. The actual hard part isn't calling an LLM, it's the prompt engineering and the safety net around it.

We're using **OpenRouter** as the model provider (because it gives access to several genuinely free models behind one API key) and **LangChain** to wrap the API call, build the prompt, and keep the conversation flowing.

A couple of things worth knowing before you run this:
- You'll need a free OpenRouter API key from [openrouter.ai/keys](https://openrouter.ai/keys). No credit card needed for the free-tier models.
- Free models on OpenRouter rotate from time to time as new ones get added and older ones get retired, so instead of hardcoding one specific model, this notebook defaults to `openrouter/free` — OpenRouter's own auto-router that picks a free model for you. If you want a specific one instead (e.g. you've tested it and like its tone), the model name is a single variable you can swap.

## 1. Install What We Need

In [1]:
# !pip install -q langchain langchain-openrouter python-dotenv

## 2. Set Up Your API Key

Either way, **don't commit a real key to a public notebook or repo.**

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

print("API key loaded.")

API key loaded.


## 3. Connect LangChain to OpenRouter

OpenRouter speaks the same API shape as OpenAI, so rather than reaching for a separate OpenRouter-specific package (which tends to lag behind in updates), we just point LangChain's regular `ChatOpenAI` class at OpenRouter's URL instead of OpenAI's. This is the integration pattern OpenRouter itself recommends, and it means we get all of LangChain's normal features — streaming, retries, etc. — for free.

`MODEL_NAME` defaults to `openrouter/free`, which auto-picks a free model for you on each request. If you'd rather pin a specific one, here are a few that are free as of mid-2026 — just swap the string:
- `meta-llama/llama-3.3-70b-instruct:free`
- `google/gemma-4-31b-it:free`
- `openai/gpt-oss-20b:free`

(Always double check [openrouter.ai/models](https://openrouter.ai/models) and filter by "free" — pricing and availability change.)

In [5]:
from langchain_openrouter import ChatOpenRouter
model_name = "openrouter/owl-alpha"
llm = ChatOpenRouter(
    model= model_name,
    max_tokens=1024)

print(f"LLM client ready")

LLM client ready


In [6]:
llm.invoke("Just say hii..")

AIMessage(content='Hii! 👋 How can I help you today?', additional_kwargs={}, response_metadata={'model_name': 'openrouter/owl-alpha', 'id': 'gen-1782129021-NRitWvvC6icnm8Lxl74O', 'created': 1782129021, 'object': 'chat.completion', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'openrouter'}, id='lc_run--019eef2a-cf06-74c3-952f-480daf26bf36-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 14, 'total_tokens': 112, 'input_token_details': {'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'reasoning': 0}})

## 4. The System Prompt — This Is Where the Real Work Happens

Honestly, this is the part of the task that actually matters. The system prompt is doing three jobs at once:

1. **Setting the persona** — friendly, plain-language, not robotic ("Act like a helpful medical assistant" from the brief, basically).
2. **Setting the scope** — general health information only, nothing that should require an actual doctor's judgment.
3. **Setting the guardrails** — refuse dosing/diagnosis/emergency stuff and redirect to a real professional instead.

We're using LangChain's `ChatPromptTemplate` so the system instructions and the live conversation history stay cleanly separated — the model always sees the same rules every single turn, no matter how long the chat gets.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

SYSTEM_PROMPT = """\
Role: Friendly, conversational health information assistant (a knowledgeable peer, not a doctor). 
Tone: Warm, jargon-free, concise, and highly skimmable (bullet points preferred). 

Capabilities: General health education, explaining wellness topics (diet, sleep), and medical definitions.

CRITICAL BOUNDARIES (Strictly Enforced):
1. NO diagnosing. Only mention general symptom associations.
2. NO specific drug dosing instructions. Provide general safety context only.
3. NO altering of prescribed treatment plans.
4. EMERGENCY PROTOCOL: For life-threatening symptoms (e.g., chest pain, breathing difficulties), IMMEDIATELY and clearly tell them to seek emergency care before any other text.
5. NO absolute or confident claims on rare conditions/drug interactions. Use cautious language ("generally," "often").

Ending: Naturally vary and include a brief reminder that this is general info, not a substitute for a doctor (skip only for trivial/factual FAQs).

"""

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{user_input}"),
])

print("Prompt template ready.")

Prompt template ready.


**Why phrase it this way instead of just "be safe":** vague instructions like "avoid harmful advice" don't actually tell the model what to do in the moment. Spelling out the specific behaviors — no dosing numbers, no diagnosing, emergency symptoms get flagged first — gives it concrete decision rules instead of a vibe to interpret. It still won't be perfect (no prompt is), which is exactly why we're adding a second layer of safety in code, not just in the prompt. More on that next.

## 5. A Safety Layer That Doesn't Rely on the Model Behaving

Here's the thing about prompt-based safety — it's a strong first layer, but it's still just an instruction the model *might* slip on, especially with a small free model under load. So we add a second, deterministic layer that runs in plain Python, before the user's message ever reaches the LLM:

- **Emergency detection** — if the message contains language suggesting a possible emergency, we skip the LLM entirely and return an immediate, consistent "please seek emergency care" message. No room for an off day from the model on something this serious.
- **Self-harm / crisis detection** — same idea, redirected to crisis resources instead.
- **A light dosing-request check** — if someone's clearly asking for an exact dose ("how many mg of X for a 5 year old"), we still let the LLM answer (it's instructed to stay general), but we append a firmer, non-negotiable safety note underneath its answer so the boundary doesn't depend solely on the model remembering its instructions.

This is the same "don't trust the model alone for the things that really matter" pattern you'd use in any production chatbot handling sensitive topics.

In [ ]:
import re

EMERGENCY_PATTERNS = [
    r"chest pain", r"can'?t breathe", r"difficulty breathing", r"struggling to breathe",
    r"trouble breathing", r"hard(?:ly)? to breathe", r"short(?:ness)? of breath",
    r"severe bleeding", r"won'?t stop bleeding", r"unconscious", r"passed out",
    r"stroke", r"face.*droop", r"slurred speech",
    r"severe allergic reaction", r"anaphyla", r"throat.*closing",
    r"overdose", r"swallowed.*pills", r"poison",
]

CRISIS_PATTERNS = [
    r"suicide", r"kill myself", r"want to die", r"end my life", r"self.?harm",
]

DOSING_REQUEST_PATTERNS = [
    r"\bhow many (mg|milligrams|ml|pills|tablets)\b",
    r"\bwhat dose\b", r"\bwhat dosage\b", r"\bhow much .*(should|can) (i|my child|my kid)\b",
]

EMERGENCY_RESPONSE = (
    "🚨 **This sounds like it could be a medical emergency.** Please call your local "
    "emergency number 1122 or get to an "
    "emergency room right now. I'm not able to safely help with this here — a real "
    "emergency responder can. Please reach out to them immediately."
)

CRISIS_RESPONSE = (
    "I'm really glad you're telling me this, and I want you to know you matter. "
    "I'm not the right resource to help with this safely, but a real person trained for "
    "You deserve support from someone who can really be there."
)

DOSING_SAFETY_NOTE = (
    "\n\n⚠️ **One important note:** I'm intentionally not giving exact dosing numbers — "
    "that really needs to come from a pharmacist, doctor, or the medicine's official "
    "package leaflet, since the right dose depends on weight, age, and other factors "
    "I don't have a safe way to verify here."
)


def screen_input(user_message: str) -> str | None:
    """
    Checks the raw user message before it ever reaches the LLM.
    Returns a hardcoded safe response if something serious is detected,
    or None if it's fine to pass along to the model.
    """
    lowered = user_message.lower()

    if any(re.search(pattern, lowered) for pattern in EMERGENCY_PATTERNS):
        return EMERGENCY_RESPONSE

    if any(re.search(pattern, lowered) for pattern in CRISIS_PATTERNS):
        return CRISIS_RESPONSE

    return None


def is_dosing_request(user_message: str) -> bool:
    lowered = user_message.lower()
    return any(re.search(pattern, lowered) for pattern in DOSING_REQUEST_PATTERNS)


# Quick sanity check that the patterns actually catch what they should
assert screen_input("I have chest pain and feel dizzy") == EMERGENCY_RESPONSE
assert screen_input("I'm having trouble breathing") == EMERGENCY_RESPONSE
assert screen_input("I want to end my life") == CRISIS_RESPONSE
assert screen_input("What causes a sore throat?") is None
assert is_dosing_request("How many mg of paracetamol for a 5 year old?") is True
assert is_dosing_request("Is paracetamol safe for children?") is False
print("Safety filter checks passed.")

Safety filter checks passed.


Quick note on that last assertion: notice "Is paracetamol safe for children?" does **not** trip the dosing filter — it's a genuinely answerable general-safety question, not a request for an exact number. That's the example query from the task brief, and it's a nice illustration of the difference: we want the bot to actually engage with these questions in a useful way, not refuse anything that mentions a medication. The line we're drawing is specifically around precise numbers, not the topic as a whole.

## 6. Wiring It Together Into an Actual Conversation

Now we connect the prompt template to the LLM and add memory, so the bot can handle natural follow-ups like "what about ibuprofen instead?" without losing the thread of what was asked before.

We're keeping this simple and explicit (a plain Python list of messages) rather than reaching for one of LangChain's heavier memory abstractions — for a chatbot this size, an explicit list is easier to read, debug, and trim than a wrapped memory object, and it's exactly what `MessagesPlaceholder` expects.

In [9]:
from langchain_core.messages import HumanMessage, AIMessage

chain = chat_prompt | llm

MAX_HISTORY_TURNS = 6  # keep the last few turns so the prompt doesn't grow unbounded

class HealthChatbot:
    def __init__(self):
        self.history = []  # list of HumanMessage / AIMessage

    def ask(self, user_message: str) -> str:
        # Layer 1: deterministic screening, before the model ever sees this message
        safe_override = screen_input(user_message)
        if safe_override:
            self.history.append(HumanMessage(content=user_message))
            self.history.append(AIMessage(content=safe_override))
            return safe_override

        # Layer 2: let the LLM answer, guided by the system prompt
        trimmed_history = self.history[-(MAX_HISTORY_TURNS * 2):]
        response = chain.invoke({"history": trimmed_history, "user_input": user_message})
        reply = response.content

        # Layer 3: reinforce the dosing boundary in code, in case the model forgets
        if is_dosing_request(user_message) and "leaflet" not in reply.lower() and "pharmacist" not in reply.lower():
            reply += DOSING_SAFETY_NOTE

        self.history.append(HumanMessage(content=user_message))
        self.history.append(AIMessage(content=reply))
        return reply

    def reset(self):
        self.history = []


chatbot = HealthChatbot()
print("Chatbot ready.")

Chatbot ready.


## 7. Trying It Out on the Brief's Example Questions

Let's run through the two example queries from the task, plus a follow-up question to confirm the conversation memory is actually working, and one deliberately risky question to check the safety layer kicks in properly.

In [10]:
test_queries = [
    "What causes a sore throat?",
    "Is paracetamol safe for children?",
    "What about ibuprofen instead?",          # follow-up, tests memory
    "I have severe chest pain right now",      # should trigger the emergency override
]

for query in test_queries:
    print(f"You: {query}")
    print(f"Bot: {chatbot.ask(query)}")
    print("-" * 80)

You: What causes a sore throat?
Bot: A sore throat is usually caused by a **viral infection**, just like the common cold or the flu. Very often, it's the first sign that you're coming down with something. 

Sometimes, a sore throat can be caused by a **bacterial infection** (like strep throat), or by irritants like dry air, allergies, pollutants, or even straining your voice too much.
--------------------------------------------------------------------------------
You: Is paracetamol safe for children?
Bot: Paracetamol is generally considered safe for children **when given at the correct dose for their weight and age**, and it is commonly used to relieve pain and reduce fever.

However, it is very important to follow these safety rules: 

- Always use the liquid medicine specifically designed for children and check the label carefully.
- 
- Give the dose recommended for the child's specific weight.
- Do not give it to children more frequently than the instructions allow.
- If a child h

What to look for when this runs: the sore throat and paracetamol answers should come back warm and easy to read, not clinical-sounding. The ibuprofen follow-up should clearly understand it's still talking about giving medicine to children — that's the memory doing its job. And the chest pain message should never reach the LLM at all; it should get the exact same hardcoded emergency message every time, instantly, regardless of how the model happens to be feeling that day.

## 8. An Actual Back-and-Forth Chat Loop

If you'd rather just talk to it directly instead of running a list of test queries, here's a simple loop. Type `quit` or `exit` to stop, or `reset` to clear the conversation history and start fresh.

In [11]:
def run_chat_loop():
    print("💬 General Health Chatbot — type 'quit' to stop, 'reset' to clear history.\n")
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() in {"quit", "exit"}:
            print("Take care! 👋")
            break
        if user_input.lower() == "reset":
            chatbot.reset()
            print("(conversation history cleared)\n")
            continue

        reply = chatbot.ask(user_input)
        print(f"Bot: {reply}\n")


# Uncomment the line below to chat interactively in this cell:
# run_chat_loop()

Left that one commented out on purpose — it's an `input()` loop, which works great when you're running this cell-by-cell yourself, but isn't something you want firing automatically if this notebook gets run top-to-bottom unattended (e.g. in CI, or by someone clicking "Run All"). Just uncomment the last line and run the cell when you're ready to actually chat.

## 9. What I Actually Built

Quick recap of how this all fits together:

- **LangChain + OpenRouter** for the model call, using the standard `ChatOpenAI` integration pointed at OpenRouter's endpoint rather than a separate niche package — keeps things stable as both ecosystems evolve.
- **A detailed system prompt** doing the heavy lifting on tone (friendly, plain-language) and the first pass at safety boundaries (no diagnosing, no exact dosing, emergencies get flagged first).
- **A code-level safety layer that runs before the LLM is even called** for anything emergency- or crisis-adjacent, so the most serious cases never depend on the model getting it right — they get a consistent, pre-written response every time.
- **Simple, explicit conversation memory** so follow-up questions work naturally.

If you wanted to take this further, a few natural next steps: swap in a paid, larger model for noticeably better answer quality; add a small eval set of test questions you re-run whenever you tweak the prompt, so you can actually tell if a change helped or hurt; or log flagged messages (emergency/crisis/dosing) somewhere so you can see how often real users are hitting those boundaries.

**Standard disclaimer, said plainly:** this is a learning project, not a medical product. It shouldn't be used as a real source of health guidance for yourself or anyone else.